In [1]:
# from importlib import reload

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pylab as pl
import torch
import torch.nn as nn
import torch.optim as optim
from matplotlib import collections as mc
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

import sys
sys.path.append("..")
from property_predictor import PropertyPredictor

plt.style.use("ggplot")
plt.style.use("seaborn-v0_8-colorblind")

In [2]:
# torch.load('../checkpoints/model.pth')

In [3]:
# load data
df = pd.read_csv(
    "../dataset/Dataset_VisContest_Rapid_Alloy_development_v3.txt",
    encoding="latin1",
    sep="\t",
)
display(df.head(2))
print("Total number of rows", len(df))

,KS1295[%],6082[%],2024[%],bat-box[%],3003[%],4032[%],Al,Si,Cu,Ni,...,Unnamed: 127,Unnamed: 128,Unnamed: 129,Unnamed: 130,Unnamed: 131,Unnamed: 132,Unnamed: 133,Unnamed: 134,Unnamed: 135,Unnamed: 136
0,0.0,0.0,0.0,0.0,0.0,100.0,83.67500,12.25000,0.900000,1.3000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.0,0.0,0.0,0.0,3.3,96.7,84.11885,11.86555,0.874425,1.2571,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Total number of rows 324632


In [4]:
# partition data by categories
n_inputs = 6
n_elements = 12

all_cols = df.columns.tolist()
input_cols = all_cols[:n_inputs]
element_cols = all_cols[n_inputs : n_inputs + n_elements]
microstructure_cols = all_cols[18 : 18 + 38]
property_cols = all_cols[56:70]
print(f"{len(input_cols)} Input scraps:", input_cols, end="\n\n")
print(f"{len(element_cols)} Elements:", element_cols, end="\n\n")
print(f"{len(microstructure_cols)} Microstructures:", microstructure_cols, end="\n\n")
print(f"{len(property_cols)} Properties:", property_cols, end="\n\n")

6 Input scraps: ['KS1295[%]', '6082[%]', '2024[%]', 'bat-box[%]', '3003[%]', '4032[%]']

12 Elements: ['Al', 'Si', 'Cu', 'Ni', 'Mg', 'Mn', 'Fe', 'Cr', 'Ti', 'Zr', 'V', 'Zn']

38 Microstructures: ['Vf_FCC_A1', 'Vf_DIAMOND_A4', 'Vf_AL15SI2M4', 'Vf_AL3X', 'Vf_AL6MN', 'Vf_MG2ZN3', 'Vf_AL3NI2', 'Vf_AL3NI_D011', 'Vf_AL7CU4NI', 'Vf_AL2CU_C16', 'Vf_Q_ALCUMGSI', 'Vf_AL7CU2FE', 'Vf_MG2SI_C1', 'Vf_AL9FE2SI2', 'Vf_AL18FE2MG7SI10', 'eut. frac.[%]', 'eut. T (°C)', 'T_FCC_A1', 'T_DIAMOND_A4', 'T_AL15SI2M4', 'T_AL3X', 'T_AL6MN', 'T_MG2ZN3', 'T_AL3NI2', 'T_AL3NI_D011', 'T_AL7CU4NI', 'T_AL2CU_C16', 'T_Q_ALCUMGSI', 'T_AL7CU2FE', 'T_MG2SI_C1', 'T_AL9FE2SI2', 'T_AL18FE2MG7SI10', 'T(liqu)', 'T(sol)', 'delta_T', 'delta_T_FCC', 'delta_T_Al15Si2M4', 'delta_T_Si']

14 Properties: ['CSC', 'YS(MPa)', 'hardness(Vickers)', 'CTEvol(1/K)(20.0-300.0°C)', 'Density(g/cm3)', 'Volume(m3/mol)', 'El.conductivity(S/m)', 'El. resistivity(ohm m)', 'heat capacity(J/(mol K))', 'Therm.conductivity(W/(mK))', 'Therm. diffusivity(m2

In [5]:
# handle na
df = df.fillna(value=0)

# # # subsample
# n = 20000  # number of samples
# df = df.sample(n)  # sample data for speed&test
# # df.to_csv(f'../dataset/data_sample_{n}.csv', index=False)
# # df

In [6]:
len(element_cols)

12

In [7]:
# Create training data
x_cols = element_cols

# property features + features that were used in the case study
y_cols = property_cols + [
    "Vf_FCC_A1",
    "delta_T",
    "T(liqu)",
    'eut. frac.[%]', # forgot to include this!
    "Vf_DIAMOND_A4",
    "Vf_AL15SI2M4",
]

X = df[x_cols].to_numpy().astype(np.float32)
y = df[y_cols].to_numpy().astype(np.float32)

# normalize
y_mean = y.mean(axis=0)
y_std = y.std(axis=0)
y = (y - y_mean) / (y_std)

X, y = torch.from_numpy(X), torch.from_numpy(y)

In [8]:
len(property_cols), y.shape

(14, torch.Size([324632, 20]))

In [9]:
print(f"{'property':>50} | {'mean':>15}  {'std':>15}")
print("-" * 120)
for name, mean, std in zip(y_cols, y_mean, y_std):
    print(f"{name:>50} : {mean:20.10f}, {std:20.10f}")

                                          property |            mean              std
------------------------------------------------------------------------------------------------------------------------
                                               CSC :         0.4562154114,         0.0637117401
                                           YS(MPa) :       277.7975158691,        41.7140769958
                                 hardness(Vickers) :        84.9859237671,        12.7541637421
                         CTEvol(1/K)(20.0-300.0°C) :         0.0000773047,         0.0000022446
                                    Density(g/cm3) :         2.6963665485,         0.0161947217
                                    Volume(m3/mol) :         0.0000102145,         0.0000000359
                              El.conductivity(S/m) :  12814342.0000000000,    679482.8125000000
                            El. resistivity(ohm m) :         0.0000000783,         0.0000000042
                         

In [10]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device(
        "mps"
    )  # Metal Performance Shaders (MPS) for Apple Silicon Macs
else:
    device = torch.device("cpu")

print("Using device:", device)

X, y = X.to(device), y.to(device)

Using device: mps


In [11]:
# model.load_state_dict(torch.load("../checkpoints/model_best.pth"))

In [12]:
def save_model(model, y_mean, y_std, x_cols, y_cols, hidden_dim, path="../checkpoints/model.pth"):
    torch.save(
        dict(
            state_dict=model.state_dict(),
            y_mean=torch.from_numpy(y_mean),
            y_std=torch.from_numpy(y_std),
            x_cols=x_cols,
            y_cols=y_cols,
            hidden_dim=hidden_dim,
        ),
        path,
    )


def max_residual(pred, truth):
    return np.abs(pred - truth).max(axis=0)

In [13]:
# Hyperparameters
hidden_dim = 1024
batch_size = 512
epochs = 200
lr = 0.001

input_dim = X.shape[1]
output_dim = y.shape[1]

# DataLoader
dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Model, loss, optimizer
model = PropertyPredictor(input_dim, hidden_dim, output_dim).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999))
# optimizer = optim.SGD(model.parameters(), lr=lr)

# training prep
rcurrent = 1e4
# test_sample = torch.randperm(y.size(0))[:3000]
test_sample = torch.randperm(y.size(0))


## Load model checkpoint or train model

### Load pretrained model

In [14]:
# model_checkpoint = torch.load("../checkpoints/best.pth", weights_only=True)
# model.load_state_dict(model_checkpoint["state_dict"])

### Train model

In [15]:
# Training loop
pbar = tqdm(range(epochs))
for epoch in pbar:
    for batch_X, batch_y in loader:
        optimizer.zero_grad()
        output = model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
    pbar.set_postfix_str(f"Epoch {epoch+1}, Loss: {loss.item():.8f}")

    if epoch % 10 == 0 or epoch == epochs-1:
        # Evaluate the model by examining the maximum residuals for each output dimension
        pred = model(X[test_sample]).detach().cpu().numpy()
        truth = y[test_sample].cpu().numpy()

        residuals = max_residual(pred, truth)
        rmax = residuals.mean()
        if rmax < rcurrent:
            # torch.save(model.state_dict(), "../checkpoints/model_best.pth")
            save_model(
                model,
                y_mean,
                y_std,
                x_cols,
                y_cols,
                hidden_dim,
                path="../checkpoints/best.pth",
            )
            rcurrent = rmax
            print('Best so far:')

        print(f"{'property':>50} | Max residuals")
        for name, res in zip(y_cols, residuals):
            print(f"{name:>50} : {res}")
        print()

  0%|          | 0/200 [00:00<?, ?it/s]

Best so far:
                                          property | Max residuals
                                               CSC : 9.007424354553223
                                           YS(MPa) : 0.7601183652877808
                                 hardness(Vickers) : 0.9548338651657104
                         CTEvol(1/K)(20.0-300.0°C) : 0.6062333583831787
                                    Density(g/cm3) : 0.9273075461387634
                                    Volume(m3/mol) : 0.8869161605834961
                              El.conductivity(S/m) : 0.8358542919158936
                            El. resistivity(ohm m) : 0.8243374824523926
                          heat capacity(J/(mol K)) : 0.5068221688270569
                        Therm.conductivity(W/(mK)) : 0.8804234266281128
                          Therm. diffusivity(m2/s) : 1.1301629543304443
                           Therm.resistivity(mK/W) : 1.0430395603179932
      Linear thermal expansion (1/K)(20.0-300.0°C) : 0.38

KeyboardInterrupt: 

## Performance Vis

In [ ]:
sample = torch.randperm(y.size(0))[:10000]
pred = model(X[sample]).detach().cpu()
truth = y[sample].cpu()

vmax = max(truth.max(), pred.max())
plt.scatter(truth.flatten(), pred.flatten(), zorder=2)
plt.plot([0, vmax], [0, vmax], c="#aaa", zorder=1, linewidth=0.5)
plt.xlabel("Ground truth")
plt.ylabel("Prediction")

# plot pred, ground truth and residuals against X

lines = list(
    zip(
        zip(X[sample, 0].cpu().numpy(), truth[:, 0].cpu().numpy()),
        zip(X[sample, 0].cpu().numpy(), pred[:, 0].detach().cpu().numpy()),
    )
)
lc = mc.LineCollection(lines, linewidths=0.5, colors="#aaa", zorder=1)
fig, ax = pl.subplots()
ax.add_collection(lc)
ax.autoscale()
ax.margins(0.1)
plt.scatter(
    X[sample, 0].cpu().numpy(), truth[:, 0].cpu().numpy(), s=5, label="ground truth"
)
plt.scatter(
    X[sample, 0].cpu().numpy(),
    pred[:, 0].detach().cpu().numpy(),
    s=5,
    c="C2",
    label="pred",
)
plt.legend()
plt.show()

## Test UI

In [ ]:
from random import randint

In [ ]:
x_min = X.min(axis=0).values
x_max = X.max(axis=0).values
x_extent = torch.stack([x_min, x_max], dim=1)

x_extent.shape

In [ ]:
def predict_local(
    model,
    X_local,
    xdim=0,
    x_cols=["Al", "Si", "Cu", "Ni", "Mg", "Mn", "Fe", "Cr", "Ti", "Zr", "V", "Zn"],
    num_points_on_curve=100,
):
    x_ref = X_local.mean(axis=0)
    r = torch.arange(len(x_cols))
    all_except_current = r != xdim

    # model prediction curve
    n = num_points_on_curve
    x_test = x_ref.repeat([n, 1])
    x_test[:, xdim] = torch.linspace(X[:, xdim].min(), X[:, xdim].max(), n)
    x_curve = x_test[:, xdim]
    with torch.no_grad():
        y_curves = model(x_test)
    return [x_curve.cpu().numpy(), y_curves.cpu().numpy()]

In [ ]:
x, ypred = predict_local(model, X_local=X[:], xdim=2, x_cols=x_cols)

plt.scatter(
    X[sample, 2].cpu().numpy(),
    y[sample, 0].cpu().numpy(),
    s=5,
    c="#333",
)
plt.scatter(x, ypred[:, 0])

In [ ]:
sample = torch.randperm(y.size(0))[:10000]  # sample points used for scatter plot
x_ref = X.mean(axis=0)
# x_ref = X[randint(0, X.shape[0])]
plt.figure(figsize=[24, 24])

for xdim in range(0, len(x_cols)):
    for ydim in range(0, len(y_cols)):

        r = torch.arange(len(x_cols))
        all_except_current = r != xdim

        close_to_ref = (
            (X[:, all_except_current] - x_ref[all_except_current])
            .abs()
            .max(axis=1)
            .values
            < 0.2
        ).cpu()
        # print(close_to_ref.sum().item())
        # close_to_ref

        plt.subplot(len(y_cols), len(x_cols), ydim * len(x_cols) + xdim + 1)

        # scatter plot
        plt.scatter(
            X[sample, xdim].cpu().numpy(),
            y[sample, ydim].cpu().numpy(),
            s=5,
            c="#333",
        )
        plt.scatter(
            X.cpu()[close_to_ref, xdim].numpy(),
            y.cpu()[close_to_ref, ydim].numpy(),
            s=5,
            label="ground truth",
        )

        # model prediction curve
        n = 100  # number of points on the curve
        x_curve = x_ref.repeat([n, 1])
        x_curve[:, xdim] = torch.linspace(X[:, xdim].min(), X[:, xdim].max(), n)
        # x_curve[:, xdim] = torch.linspace(
        #     X[close_to_ref, xdim].min(), X[close_to_ref, xdim].max(), n
        # )
        with torch.no_grad():
            y_curve = model(x_curve)
        plt.plot(
            x_curve[:, xdim].cpu().numpy(), y_curve[:, ydim].cpu().numpy(), c="orange"
        )
        if ydim == len(y_cols) - 1:
            plt.xlabel(x_cols[xdim])
        if xdim == 0:
            plt.ylabel(y_cols[ydim])
plt.show()